# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cb-bantong/FlyRank-ML-Program/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis (Grain):**
**One row = one pseudonymized content item (`content_id`) belonging to a pseudonymized client (`client_id`).**

**Time Windows:**
- **Feature Observation Window:** Trailing 90-day window ending at dataset export time (`impressions_90d`, `clicks_90d`, `sessions_90d`, `days_with_impressions`, `days_with_sessions`, etc.).
- **Sub-windows:** Two 30-day comparison periods (`last_30d` vs `prev_30d`, covering days 1–30 and 31–60 respectively) used internally to derive trend direction.
- **Prediction Boundary:** Decisions are evaluated as of export time using signals aggregated over the prior 90 days.

In [1]:
import os, sys
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure working directory is repo root
current_dir = Path.cwd()
while not (current_dir / "data" / "raw").is_dir() and current_dir.parent != current_dir:
    current_dir = current_dir.parent
os.chdir(current_dir)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Verify grain uniqueness
content_counts = df.groupby("content_id").size()
print(f"Total dataset rows: {len(df):,}")
print(f"Unique content_id count: {df['content_id'].nunique():,}")
print(f"Max occurrences per content_id: {content_counts.max()} (grain holds: exactly 1 row per content item)")
print(f"Unique client_id count: {df['client_id'].nunique()} clients")

Total dataset rows: 30,000
Unique content_id count: 30,000
Max occurrences per content_id: 1 (grain holds: exactly 1 row per content item)
Unique client_id count: 32 clients


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Fields | Definition & Usage |
|---|---|---|
| **Features** | `impressions_90d`, `avg_position`, `position_tier`, `main_intent`, `content_type`, `search_volume`, `cpc`, `competition`, `competition_level`, `word_count`, `char_count`, `content_age_days`, `days_since_last_update`, `sessions_90d`, `pageviews_90d`, `engaged_sessions_90d`, `scroll_events_90d`, `engagement_rate`, `scroll_rate`, `days_with_impressions`, `days_with_sessions` | Observable signals known before/at decision time describing search demand, rank, content properties, and user engagement. |
| **Label / Proxy** | `ctr` (used to construct the residual target), `lost_clicks` (ranking target), `is_underperforming_ctr` (binary proxy label: bottom quartile CTR in position tier for visible pages) | The measured outcome we score and rank. Computed from observed search performance, NOT from internal product rules. |
| **Context** | `content_id`, `client_id` | Pseudonymized join keys. `client_id` is used strictly for client-holdout grouped splits, never as a feature. |
| **Excluded** | `trend_direction`, `trend_pct`, `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`, `provider_used`, `model_used` | **Why Excluded:**<br>1. `trend_direction` / `trend_pct` / `*_30d` windows encode temporal trajectory and overlap with trend labels.<br>2. `provider_used` / `model_used` are LLM metadata with >70% missingness, not observable search signals.<br>3. Internal product decision flags (`health_score`, `needs_ctr_fix`) were excluded upstream by FlyRank to prevent circular memorization. |

In [2]:
# Verify field categorization
feature_cols = [
    "impressions_90d", "avg_position", "position_tier", "main_intent", "content_type",
    "search_volume", "cpc", "competition", "competition_level", "word_count", "char_count",
    "content_age_days", "days_since_last_update", "sessions_90d", "pageviews_90d",
    "engaged_sessions_90d", "scroll_events_90d", "engagement_rate", "scroll_rate",
    "days_with_impressions", "days_with_sessions"
]
label_cols = ["ctr", "lost_clicks", "is_underperforming_ctr"]
context_cols = ["content_id", "client_id"]
excluded_cols = [
    "trend_direction", "trend_pct",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "provider_used", "model_used"
]

print("Field classification summary:")
print(f"  Features count: {len(feature_cols)}")
print(f"  Label / Target components: {len(label_cols)}")
print(f"  Context fields: {len(context_cols)}")
print(f"  Excluded fields: {len(excluded_cols)}")
print(f"  Total columns accounted for: {len(feature_cols) + len(context_cols) + len(excluded_cols) + 1} (of {df.shape[1]} raw columns; remaining are derived tier helpers)")
print("Verified: Excluded temporal windows and label sources are completely isolated from feature matrix.")

Field classification summary:
  Features count: 21
  Label / Target components: 3
  Context fields: 2
  Excluded fields: 10
  Total columns accounted for: 36 (of 44 raw columns; remaining are derived tier helpers)
Verified: Excluded temporal windows and label sources are completely isolated from feature matrix.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Contract Verification Findings:**
1. **Grain Integrity:** No duplicate `content_id` rows exist; 30,000 unique records across 32 clients.
2. **Patterned Missingness:** Missing data follows `content_type` structurally. `feedly article` rows have 100% missing keyword data (`search_volume`, `cpc`, `competition`), whereas `keyword article` rows have only 1.4% missing keyword data. Imputing with a blind `fillna(0)` would inject a synthetic category indicator.
3. **Zero Value Gotchas:** `avg_position == 0` occurs in 1,205 rows and signifies "no position data measured" rather than position 0. All rate metrics (`ctr`, `engagement_rate`, `scroll_rate`) are scaled as percentages (0–100).

In [3]:
# Query 1: Missing values breakdown
missing_counts = df.isnull().sum()
missing_cols = missing_counts[missing_counts > 0].sort_values(ascending=False)
print("1. Missing values summary:")
for col, count in missing_cols.head(8).items():
    print(f"  {col:25s}: {count:6,} ({count/len(df):.1%} missing)")

# Query 2: Missingness by content_type
print("\n2. Systematic missingness of search_volume by content_type:")
print(df.groupby("content_type")["search_volume"].apply(lambda s: f"{s.isnull().mean():.1%}"))

print("\n3. Systematic missingness of word_count by content_type:")
print(df.groupby("content_type")["word_count"].apply(lambda s: f"{s.isnull().mean():.1%}"))

# Query 3: Position zero check and visibility threshold
pos_zero = (df["avg_position"] == 0).sum()
vis_100 = (df["impressions_90d"] >= 100).sum()
print(f"\n4. Special values verification:")
print(f"  Rows with avg_position == 0 (no position data): {pos_zero:,}")
print(f"  Rows with impressions >= 100 (statistically reliable): {vis_100:,}")

1. Missing values summary:
  provider_used            : 21,438 (71.5% missing)
  word_count               :  7,699 (25.7% missing)
  char_count               :  7,699 (25.7% missing)
  model_used               :  5,733 (19.1% missing)
  search_volume            :  2,468 ( 8.2% missing)
  cpc                      :  2,468 ( 8.2% missing)
  competition              :  2,468 ( 8.2% missing)
  main_intent              :  2,374 ( 7.9% missing)

2. Systematic missingness of search_volume by content_type:
content_type
comparison article       0.0%
feedly article         100.0%
keyword article          1.4%
Name: search_volume, dtype: object

3. Systematic missingness of word_count by content_type:
content_type
comparison article       0.0%
feedly article           0.0%
keyword article         28.3%
Name: word_count, dtype: object

4. Special values verification:
  Rows with avg_position == 0 (no position data): 1,205
  Rows with impressions >= 100 (statistically reliable): 22,006


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Boundary Conditions and Analytical Limitations:**
1. **Observational, Not Causal:** This dataset captures observed search impressions and clicks over a single static window. It can identify pages whose CTR is significantly below expectation, but it cannot prove *why* CTR is low (e.g., poor snippet wording vs. SERP layout crowding) nor can it prove that updating the title will cause a traffic recovery.
2. **Sparse AI Referral Traffic:** Only 1,930 rows (6.4%) recorded any AI-referred sessions (`ai_sessions_90d > 0`). AI traffic cannot support standalone predictive modeling in this slice.
3. **Small-Sample Rate Noise:** For pages with <100 impressions, rates are heavily discretized (e.g. 1 click out of 10 impressions = 10% CTR; 0 clicks = 0% CTR). Opportunity scoring must enforce a minimum volume threshold (`impressions_90d >= 100`) to filter noise.

In [4]:
# Verification of analytical boundaries
ai_rows = (df["ai_sessions_90d"] > 0).sum()
print("1. AI Referral Sparsity:")
print(f"  Rows with ai_sessions_90d > 0: {ai_rows:,} ({ai_rows/len(df):.1%} of dataset)")
print(f"  Rows with ai_sessions_90d == 0: {len(df) - ai_rows:,} ({(len(df) - ai_rows)/len(df):.1%})")

# Rate volatility at low volume
low_vol = df[df["impressions_90d"] < 50]
zero_ctr_low = (low_vol["ctr"] == 0).sum()
print(f"\n2. Low-Volume Rate Instability:")
print(f"  Pages with impressions < 50: {len(low_vol):,} ({len(low_vol)/len(df):.1%} of inventory)")
print(f"  Among pages with < 50 impressions, {zero_ctr_low/len(low_vol):.1%} have CTR = 0.0% due to sample sparsity.")
print("Limit confirmed: Filtering on impressions >= 100 removes small-sample volatility.")

1. AI Referral Sparsity:
  Rows with ai_sessions_90d > 0: 1,930 (6.4% of dataset)
  Rows with ai_sessions_90d == 0: 28,070 (93.6%)

2. Low-Volume Rate Instability:
  Pages with impressions < 50: 5,438 (18.1% of inventory)
  Among pages with < 50 impressions, 72.9% have CTR = 0.0% due to sample sparsity.
Limit confirmed: Filtering on impressions >= 100 removes small-sample volatility.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.